In [1]:
"""SESSION 18 YOLO11l-seg: 5-fold CV + multi-seed + WBF fold ensemble.
"""
from __future__ import annotations
import multiprocessing

import argparse
import json
import random
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np

try:
    import yaml
except ImportError:
    import subprocess as _sp
    _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml

SESSION_NAME   = "session18_yolo11l"
MODEL_ARCH     = "yolo11l-seg.pt"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ---------------------------------------------------------------------------
# Auto-install helpers
# ---------------------------------------------------------------------------
def _pip(pkg: str) -> None:
    import subprocess as _sp
    print(f"  [install] {pkg} ...")
    _sp.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


def _ensure_deps() -> None:
    try:
        import ultralytics          # noqa: F401
    except ImportError:
        _pip("ultralytics")
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold  # noqa: F401
    except ImportError:
        _pip("iterative-stratification")
    try:
        from pycocotools.coco import COCO   # noqa: F401
    except ImportError:
        _pip("pycocotools")


# ---------------------------------------------------------------------------
# GPU / device
# ---------------------------------------------------------------------------
def _auto_device() -> str:
    try:
        import torch
        n = torch.cuda.device_count()
        if n >= 2:
            print(f"  [device] {n} GPUs -> '0,1' (DDP)")
            return "0,1"
        if n == 1:
            print("  [device] 1 GPU -> '0'")
            return "0"
    except Exception:
        pass
    print("  [device] no GPU -> 'cpu'")
    return "cpu"


def _single(device: str) -> str:
    """First device only - val/predict don't support DDP."""
    return str(device).split(",")[0].strip() if device not in ("cpu", "") else "cpu"


def _free_gpu_memory() -> None:
    """Aggressively release GPU memory. Called between phases/folds so that
    leftover allocations from a just-finished training run (trainer,
    optimizer, dataloader workers, etc.) can't accumulate and starve the
    next model.predict() call. This is a real fix, not just a formality:
    the OOM in Phase B reported 5.02 GiB already in use before prediction
    even started, memory that Phase A's training should have released."""
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except Exception:
        pass


# ---------------------------------------------------------------------------
# Dataset root resolution
# ---------------------------------------------------------------------------
def _find_dataset_root(hint: str) -> Path:
    """
    Resolution order (when hint == 'auto' or path not found):
      1. Any dir named 'Image_Annotation_rebuilt' under /kaggle/input  (Kaggle dataset)
      2. Any dir under /kaggle/input with train/valid/test image sub-dirs
      3. Common local paths (./Image_Annotation_rebuilt, ../Image_Annotation_rebuilt)
      4. Explicit --data-root path (only when hint != 'auto')
    Raise with a clear fix message if nothing works.
    """
    if hint != "auto":
        p = Path(hint).resolve()
        if (p / "data.yaml").is_file():
            print(f"  [data-root] using explicit path: {p}")
            return p
        print(f"  [data-root] explicit path not found: {p} - falling back to auto-detect")

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cand in sorted(kaggle_input.rglob("Image_Annotation_rebuilt")):
            if cand.is_dir() and (cand / "data.yaml").is_file():
                print(f"  [auto-detect] found named dataset at: {cand}")
                return cand.resolve()
        for dy in sorted(kaggle_input.rglob("data.yaml")):
            r = dy.parent
            if all((r / s / "images").is_dir() for s in ("train", "valid", "test")):
                print(f"  [auto-detect] found dataset layout at: {r}")
                return r.resolve()

    local_candidates = [
        Path("Image_Annotation_rebuilt"),
        Path("../Image_Annotation_rebuilt"),
        Path.home() / "Image_Annotation_rebuilt",
    ]
    for cand in local_candidates:
        cand = cand.resolve()
        if (cand / "data.yaml").is_file():
            print(f"  [auto-detect] found local dataset at: {cand}")
            return cand

    raise SystemExit(
        "\n=== SESSION 18 ERROR: Image_Annotation_rebuilt not found ===\n"
        "Auto-detection searched:\n"
        "  /kaggle/input/**/Image_Annotation_rebuilt\n"
        "  /kaggle/input/**/<any folder with train/valid/test>\n"
        "  ./Image_Annotation_rebuilt  ../Image_Annotation_rebuilt\n"
        "\nFix options:\n"
        "  A) On Kaggle: attach Image_Annotation_rebuilt as a Dataset input.\n"
        "  B) Anywhere : pass --data-root /full/path/to/Image_Annotation_rebuilt\n"
        "=============================================================="
    )


# ---------------------------------------------------------------------------
# Dataset loading
# ---------------------------------------------------------------------------
def _load_names(data_yaml: Path) -> list[str]:
    data = yaml.safe_load(data_yaml.read_text())
    names = data.get("names")
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    if isinstance(names, list):
        return [str(n) for n in names]
    raise ValueError("data.yaml: 'names' must be a list or mapping.")


def _gather_split(root: Path, split: str) -> list[dict[str, Any]]:
    """Return [{img, stem, label}] for images that have a non-empty label file."""
    img_dir = root / split / "images"
    lbl_dir = root / split / "labels"
    if not img_dir.is_dir():
        return []
    items = []
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.is_file() and lbl.read_text().strip():
            items.append({"img": img, "stem": img.stem, "label": lbl})
    return items


# ---------------------------------------------------------------------------
# Multilabel-stratified k-fold  (+  greedy fallback  +  class-presence repair)
# ---------------------------------------------------------------------------
def _multi_hot(items: list[dict], n_classes: int) -> np.ndarray:
    Y = np.zeros((len(items), n_classes), dtype=int)
    for i, it in enumerate(items):
        for line in it["label"].read_text().splitlines():
            if line.strip():
                c = int(line.split()[0])
                if 0 <= c < n_classes:
                    Y[i, c] = 1
    return Y


def _stratified_kfold(Y: np.ndarray, k: int, seed: int) -> np.ndarray:
    n = len(Y)
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
        mskf = MultilabelStratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        fold_of = np.empty(n, dtype=int)
        for f, (_, val_idx) in enumerate(mskf.split(np.zeros((n, 1)), Y)):
            fold_of[val_idx] = f
        return fold_of
    except Exception as err:
        print(f"  [warn] iterative-stratification error ({err}); greedy fallback")
        rng    = random.Random(seed)
        freq   = Y.sum(axis=0)
        rarity = Y @ (1.0 / np.maximum(freq, 1))
        order  = sorted(range(n), key=lambda i: (-rarity[i], rng.random()))
        counts = np.zeros(k)
        fold_of = np.zeros(n, dtype=int)
        for i in order:
            f = int(np.argmin(counts))
            fold_of[i] = f
            counts[f] += 1
        return fold_of


def _repair_presence(fold_of: np.ndarray, Y: np.ndarray, k: int) -> np.ndarray:
    """Guarantee every class appears in every fold validation set."""
    K = Y.shape[1]
    members: list[np.ndarray] = [np.where(fold_of == f)[0] for f in range(k)]
    changed, guard = True, 0
    while changed and guard < 300:
        changed, guard = False, guard + 1
        for f in range(k):
            if len(members[f]) == 0:
                continue
            present = Y[members[f]].sum(axis=0)
            for c in range(K):
                if present[c] > 0:
                    continue
                best_src, best_idx = -1, -1
                for d in range(k):
                    if d == f:
                        continue
                    c_mask = Y[members[d], c] > 0
                    if int(c_mask.sum()) > 1:
                        best_src  = d
                        best_idx  = int(members[d][c_mask][0])
                        break
                if best_src == -1:
                    continue
                fold_of[best_idx] = f
                members[best_src] = members[best_src][members[best_src] != best_idx]
                members[f]        = np.append(members[f], best_idx)
                present[c]        = 1
                changed           = True
    return fold_of


def _write_fold_files(work: Path, items: list[dict], fold_of: np.ndarray,
                      k: int, names: list[str]) -> list[Path]:
    yamls = []
    for f in range(k):
        fdir = work / "folds" / f"fold_{f}"
        fdir.mkdir(parents=True, exist_ok=True)
        val_idx = np.where(fold_of == f)[0]
        tr_idx  = np.where(fold_of != f)[0]
        (fdir / "train.txt").write_text(
            "\n".join(str(items[i]["img"].resolve()) for i in tr_idx) + "\n")
        (fdir / "valid.txt").write_text(
            "\n".join(str(items[i]["img"].resolve()) for i in val_idx) + "\n")
        dy = {
            "train": str((fdir / "train.txt").resolve()),
            "val":   str((fdir / "valid.txt").resolve()),
            "nc":    len(names),
            "names": {i: n for i, n in enumerate(names)},
        }
        (fdir / "data.yaml").write_text(yaml.safe_dump(dy, sort_keys=False))
        yamls.append(fdir / "data.yaml")
    return yamls


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def _train_fold(model_arch: str, data_yaml: Path,
                run_name: str, args, epochs_override: int | None = None) -> dict[str, Any]:
    from ultralytics import YOLO
    run_dir = Path(args.work_dir) / "runs" / run_name
    best_pt = run_dir / "weights" / "best.pt"
    epochs_to_run = args.epochs if epochs_override is None else epochs_override

    if args.skip_existing and best_pt.is_file():
        print(f"  [skip] {run_name}: best.pt found, re-validating only")
    else:
        model = YOLO(model_arch)
        model.train(
            data       = str(data_yaml),
            epochs     = epochs_to_run,
            imgsz      = args.imgsz,
            batch      = args.batch,
            device     = args.device,
            seed       = args.seed_for_run,
            project    = str(Path(args.work_dir) / "runs"),
            name       = run_name,
            exist_ok   = True,
            patience   = args.patience,
            cache      = args.cache,
            close_mosaic   = args.close_mosaic,
            copy_paste     = args.copy_paste,
            mosaic         = args.mosaic,
            mixup          = args.mixup,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            fliplr=0.5,  flipud=0.0,
            degrees        = args.degrees,
            translate=0.1, scale=0.5,
            optimizer      = "AdamW",
            lr0=0.001, weight_decay=0.0005,
            label_smoothing = args.label_smoothing,
            fraction       = 0.85,
            verbose=False, plots=False,
        )
        # Drop the trainer/model reference the moment training is done and
        # force a GPU memory sweep, so residual allocations from this fold
        # (optimizer states, dataloader workers, EMA copy, etc.) can't carry
        # over into the next fold's training or into Phase B's prediction
        # step. This is what the earlier OOM log's "5.02 GiB memory in use"
        # (measured before Phase B had even started) points to.
        del model
        _free_gpu_memory()

    model   = YOLO(str(best_pt))
    metrics = model.val(data=str(data_yaml), split="val",
                        imgsz=args.imgsz, device=_single(args.device),
                        rect=False, verbose=False)
    rd  = getattr(metrics, "results_dict", {}) or {}
    seg = getattr(metrics, "seg", None)
    map50   = float(rd.get("metrics/mAP50(M)",    getattr(seg, "map50", float("nan"))))
    map5095 = float(rd.get("metrics/mAP50-95(M)", getattr(seg, "map",   float("nan"))))
    per_class: dict[str, float] = {}
    try:
        for ci, c in enumerate(metrics.seg.ap_class_index):
            per_class[str(int(c))] = float(metrics.seg.ap50[ci])
    except Exception:
        pass
    del model
    _free_gpu_memory()
    return {"best_pt": str(best_pt), "val_map50": map50,
            "val_map50_95": map5095, "per_class_map50": per_class}


# ---------------------------------------------------------------------------
# Prediction + WBF mask fusion
# ---------------------------------------------------------------------------
def _predict(best_pt: str, items: list[dict], args) -> dict[str, list]:
    """Run inference; return stem -> list of dets."""
    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    from ultralytics import YOLO
    model   = YOLO(best_pt)
    out: dict[str, list] = {}
    sources = [str(it["img"].resolve()) for it in items]
    stems   = [it["stem"]              for it in items]
    try:
        results = model.predict(
            source=sources, imgsz=args.imgsz,
            conf=args.conf, iou=args.pred_iou,
            device=_single(args.device),
            augment=False,
            # batch=1: predict one image at a time. The Phase B crash was an
            # attempt to allocate 25 GiB on a 14.56 GiB card during warmup,
            # nearly double total capacity - forcing batch=1 here removes
            # batch size as a variable in that allocation, regardless of
            # what default Ultralytics would otherwise have chosen.
            batch=1,
            # half=True: FP16 inference roughly halves activation memory
            # for the same input, cutting risk of hitting this ceiling again.
            half=True,
            retina_masks=True, stream=True, verbose=False,
        )
        for stem, r in zip(stems, results):
            dets = []
            W, H = r.orig_shape[1], r.orig_shape[0]
            if r.masks is not None and r.boxes is not None and len(r.boxes) > 0:
                boxes  = r.boxes.xyxy.cpu().numpy()
                scores = r.boxes.conf.cpu().numpy()
                clss   = r.boxes.cls.cpu().numpy().astype(int)
                polys  = r.masks.xy
                for b, s, c, p in zip(boxes, scores, clss, polys):
                    if p is None or len(p) < 3:
                        continue
                    dets.append({"box": b.tolist(), "score": float(s), "cls": int(c),
                                "poly": [(float(x), float(y)) for x, y in p],
                                "wh": (int(W), int(H))})
            out[stem] = dets
    except torch.cuda.OutOfMemoryError as e:
        # Don't let one fold's OOM kill the whole ensemble run. Free memory,
        # retry once at half the image size (masks/boxes still get rescaled
        # correctly against orig_shape regardless of inference imgsz), and
        # only give up on this fold if that also fails.
        print(f"  [warn] OOM predicting with {best_pt} at imgsz={args.imgsz}: {e}")
        print(f"  [warn] retrying once at imgsz={args.imgsz // 2}, batch=1, half=True")
        del model
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        model = YOLO(best_pt)
        results = model.predict(
            source=sources, imgsz=args.imgsz // 2,
            conf=args.conf, iou=args.pred_iou,
            device=_single(args.device),
            augment=False, batch=1, half=True,
            retina_masks=True, stream=True, verbose=False,
        )
        for stem, r in zip(stems, results):
            dets = []
            W, H = r.orig_shape[1], r.orig_shape[0]
            if r.masks is not None and r.boxes is not None and len(r.boxes) > 0:
                boxes  = r.boxes.xyxy.cpu().numpy()
                scores = r.boxes.conf.cpu().numpy()
                clss   = r.boxes.cls.cpu().numpy().astype(int)
                polys  = r.masks.xy
                for b, s, c, p in zip(boxes, scores, clss, polys):
                    if p is None or len(p) < 3:
                        continue
                    dets.append({"box": b.tolist(), "score": float(s), "cls": int(c),
                                "poly": [(float(x), float(y)) for x, y in p],
                                "wh": (int(W), int(H))})
            out[stem] = dets
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return out


def _iou_xyxy(a: list, b: list) -> float:
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    inter = max(0.0, ix2-ix1) * max(0.0, iy2-iy1)
    ua    = (a[2]-a[0]) * (a[3]-a[1])
    ub    = (b[2]-b[0]) * (b[3]-b[1])
    union = ua + ub - inter
    return inter / union if union > 0 else 0.0


def _rasterise(poly: list, W: int, H: int) -> np.ndarray:
    from PIL import Image, ImageDraw
    im = Image.new("L", (W, H), 0)
    ImageDraw.Draw(im).polygon(poly, outline=1, fill=1)
    return np.asarray(im, dtype=np.uint8)


def _fuse_image(dets_per_model: list[list[dict]],
                n_models: int, args) -> list[dict]:
    """WBF-style fusion across fold models for one image."""
    alld = [d for dets in dets_per_model for d in dets]
    if not alld:
        return []
    alld.sort(key=lambda d: d["score"], reverse=True)
    used  = [False] * len(alld)
    fused = []
    for i in range(len(alld)):
        if used[i]:
            continue
        cluster = [alld[i]]; used[i] = True
        for j in range(i+1, len(alld)):
            if used[j] or alld[j]["cls"] != alld[i]["cls"]:
                continue
            if _iou_xyxy(alld[i]["box"], alld[j]["box"]) >= args.wbf_iou:
                cluster.append(alld[j]); used[j] = True
        if len(cluster) < args.min_support:
            continue
        w      = np.array([d["score"] for d in cluster])
        fbox   = (np.array([d["box"] for d in cluster]) * w[:, None]).sum(0) / w.sum()
        fscore = float(w.sum() / n_models)
        W, H   = cluster[0]["wh"]
        acc    = np.zeros((H, W), dtype=np.float32)
        for d, wi in zip(cluster, w):
            acc += float(wi) * _rasterise(d["poly"], W, H).astype(np.float32)
        soft   = acc / w.sum()
        mask   = (soft >= 0.5).astype(np.uint8)
        if mask.sum() == 0:
            continue
        fused.append({"box": fbox.tolist(), "score": min(1.0, fscore),
                      "cls": cluster[0]["cls"], "mask": mask, "wh": (W, H)})
    return fused


# ---------------------------------------------------------------------------
# COCO ground-truth / detection serialisation + eval
# ---------------------------------------------------------------------------
def _build_coco_gt(items: list[dict], names: list[str],
                   dest: Path) -> dict[str, int]:
    from PIL import Image as PImage
    images, annotations, ann_id = [], [], 1
    stem_to_id: dict[str, int] = {}
    for img_id, it in enumerate(items, start=1):
        with PImage.open(it["img"]) as im:
            W, H = im.size
        stem_to_id[it["stem"]] = img_id
        images.append({"id": img_id, "file_name": it["img"].name,
                        "width": W, "height": H})
        for line in it["label"].read_text().splitlines():
            v = line.split()
            if len(v) < 7 or (len(v)-1) % 2:
                continue
            c      = int(v[0])
            coords = [float(x) for x in v[1:]]
            xs     = [min(1.0, max(0.0, coords[i]))*W for i in range(0, len(coords), 2)]
            ys     = [min(1.0, max(0.0, coords[i]))*H for i in range(1, len(coords), 2)]
            seg    = [x for pair in zip(xs, ys) for x in pair]
            area   = 0.5 * abs(sum(
                xs[i]*ys[(i+1)%len(xs)] - xs[(i+1)%len(xs)]*ys[i]
                for i in range(len(xs))))
            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
            if area <= 1 or x1 <= x0 or y1 <= y0:
                continue
            annotations.append({"id": ann_id, "image_id": img_id,
                                 "category_id": c+1, "segmentation": [seg],
                                 "area": float(area),
                                 "bbox": [x0, y0, x1-x0, y1-y0], "iscrowd": 0})
            ann_id += 1
    coco = {"images": images, "annotations": annotations,
            "categories": [{"id": i+1, "name": n, "supercategory": "skin"}
                           for i, n in enumerate(names)]}
    dest.write_text(json.dumps(coco))
    return stem_to_id


def _dets_to_coco(pred_per_stem: dict[str, list], stem_to_id: dict[str, int],
                  dest: Path) -> Path:
    from pycocotools import mask as maskutil
    results = []
    for stem, dets in pred_per_stem.items():
        img_id = stem_to_id.get(stem)
        if img_id is None:
            continue
        for d in dets:
            m   = np.asfortranarray(d["mask"].astype(np.uint8))
            rle = maskutil.encode(m)
            rle["counts"] = rle["counts"].decode("ascii")
            results.append({"image_id": img_id, "category_id": d["cls"]+1,
                            "score": float(d["score"]), "segmentation": rle})
    dest.write_text(json.dumps(results))
    return dest


def _coco_eval(gt_json: Path, dt_json: Path,
               names: list[str]) -> dict[str, Any]:
    from pycocotools.coco     import COCO
    from pycocotools.cocoeval import COCOeval
    coco_gt = COCO(str(gt_json))
    if dt_json.stat().st_size <= 2 or json.loads(dt_json.read_text()) == []:
        return {"mAP50": 0.0, "mAP50_95": 0.0, "per_class_mAP50": {}}
    coco_dt = coco_gt.loadRes(str(dt_json))
    ev = COCOeval(coco_gt, coco_dt, "segm")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    per_class: dict[str, float] = {}
    for ci, name in enumerate(names, start=1):
        e = COCOeval(coco_gt, coco_dt, "segm")
        e.params.catIds = [ci]
        e.evaluate(); e.accumulate(); e.summarize()
        per_class[name] = float(e.stats[1])
    return {"mAP50": float(ev.stats[1]), "mAP50_95": float(ev.stats[0]),
            "per_class_mAP50": per_class}


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> int:
    ap = argparse.ArgumentParser(
        description=(
            "SESSION 18 - YOLO11l-seg 5-fold CV + WBF ensemble on "
            "Image_Annotation_rebuilt (clean splits, zero leakage). "
            "No arguments required for Kaggle T4x2 Save and Run All."
        )
    )
    ap.add_argument("--data-root",  default="auto",
                    help=("'auto' (default) = scan /kaggle/input/ and common local paths. "
                          "Pass an explicit path to override."))
    ap.add_argument("--work-dir",   default=f"/kaggle/working/{SESSION_NAME}")
    ap.add_argument("--model",        default=MODEL_ARCH)
    ap.add_argument("--folds",        type=int,   default=5,
                    help="Restored to 5 (from the 3-fold shrink) to reproduce the "
                         "0.727 mAP50 CV result. Fitting this in 12h assumes a 2-GPU "
                         "'GPU T4 x2' accelerator with working DDP - see the runtime "
                         "GPU-count check below for what happens if only 1 GPU is found.")
    ap.add_argument("--epochs",       type=int,   default=48,
                    help="Restored to 48 (from 30) to reproduce the 0.727 mAP50 result. "
                         "See --folds help and --phaseD-epochs for the full timing plan.")
    ap.add_argument("--phaseD-epochs", type=int,  default=12,
                    help="Upper bound on Phase D's fine-tune epochs. The actual number "
                         "used may be reduced below this (or Phase D skipped entirely) "
                         "based on how much of --session-budget-min is left after Phase A "
                         "and Phase B actually finish - see --session-budget-min.")
    ap.add_argument("--session-budget-min", type=float, default=690.0,
                    help="Overall wall-clock budget in minutes (Kaggle's hard limit is "
                         "720 = 12h; 690 leaves a 30 min safety margin). Phase A always "
                         "runs in full since it's what produces the CV mAP50 number. "
                         "Phase B and Phase D each check remaining budget before starting "
                         "and skip or shrink themselves rather than risk exceeding this.")
    ap.add_argument("--phaseB-reserve-min", type=float, default=25.0,
                    help="Minimum remaining minutes required to attempt Phase B at all; "
                         "below this, Phase B is skipped rather than started and risk "
                         "running past the session limit.")
    ap.add_argument("--phaseD-reserve-min", type=float, default=15.0,
                    help="Minutes reserved for Phase D's post-training predict/eval step "
                         "(not counted against its training epoch budget).")
    ap.add_argument("--imgsz",        type=int,   default=640)
    ap.add_argument("--batch",        type=int,   default=0,   help="0 = auto (16/GPU)")
    ap.add_argument("--device",       default="auto")
    ap.add_argument("--patience",     type=int,   default=20)
    ap.add_argument("--cache",        default="disk")
    ap.add_argument("--cache-dir",    default="/kaggle/working/ultralytics_cache",
                    help="Where to store disk cache (must be writable).")
    ap.add_argument("--close-mosaic", type=int,   default=10)
    ap.add_argument("--copy-paste",   type=float, default=0.5,
                    help="Copy-paste probability; 0.5 helps rare classes (SJS-TEN, Nail_psoriasis).")
    ap.add_argument("--mosaic",       type=float, default=1.0)
    ap.add_argument("--degrees",      type=float, default=0.0)
    ap.add_argument("--mixup",        type=float, default=0.1,
                    help="Mixup alpha. 0.1 = light blending; 0=off.")
    ap.add_argument("--label-smoothing", type=float, default=0.1,
                    help="Label smoothing for noisy/coarse annotations. 0.1 recommended.")
    ap.add_argument("--cv-seed",      type=int,   default=42)
    ap.add_argument("--seeds",        type=int,   nargs="+", default=[42],
                    help="Extra seeds trigger Phase C. e.g. --seeds 42 123 777")
    ap.add_argument("--conf",         type=float, default=0.001)
    ap.add_argument("--pred-iou",     type=float, default=0.7)
    ap.add_argument("--wbf-iou",      type=float, default=0.55)
    ap.add_argument("--min-support",  type=int,   default=1,
                    help="Min fold models that must agree for a detection to survive WBF.")
    ap.add_argument("--skip-existing", action="store_true", default=True,
                    help="Reuse best.pt if found (safe resume across interruptions).")
    ap.add_argument("--no-skip-existing", dest="skip_existing", action="store_false")
    ap.add_argument("--skip-train",   action="store_true",
                    help="Skip all training; ensemble from saved fold models.")
    ap.add_argument("--quick",        action="store_true",
                    help="Smoke test: 3 epochs, 300 trainval, 100 test images.")
    args, _ = ap.parse_known_args()   # ignore Jupyter/papermill injected args
    t_script_start = time.time()

    def _elapsed_min() -> float:
        return (time.time() - t_script_start) / 60.0

    multiprocessing.freeze_support()

    import os
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
    print(f"  [mem] PYTORCH_ALLOC_CONF = {os.environ['PYTORCH_ALLOC_CONF']}")

    _ensure_deps()

    if args.device == "auto":
        args.device = _auto_device()
    n_gpus = len(str(args.device).split(",")) if args.device not in ("cpu", "") else 0
    if n_gpus < 2:
        print(
            f"\n  [warn] Only {n_gpus} GPU detected/selected (device='{args.device}').\n"
            f"  The default folds=5, epochs=48 here was chosen assuming a 2-GPU 'GPU T4 x2'\n"
            f"  accelerator with DDP training, based on your last run's measured ~159s/epoch\n"
            f"  on a single GPU (5 folds x 48 epochs = 38165s ~= 10.6h for Phase A alone).\n"
            f"  With only {n_gpus} GPU, Phase A alone will likely take that same ~10.6h again,\n"
            f"  leaving too little time for Phase B/D in a 12h session.\n"
            f"  Fix: in Kaggle, go to Settings -> Accelerator and select 'GPU T4 x2'.\n"
            f"  If 2 GPUs truly aren't available to you, pass --folds 3 --epochs 30 instead.\n"
        )
    if args.batch == 0:
        args.batch = max(8, 8 * max(1, n_gpus))
        print(f"  [batch] auto -> {args.batch} ({max(1, n_gpus)} GPU(s) x 8 - OOM-safe for T4)")
    if args.cache == "False":
        args.cache = False
    if args.cache == "disk":
        cache_dir = Path(args.cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        os.environ["YOLO_CONFIG_DIR"] = str(cache_dir)
        os.environ["YOLO_DATASETS_DIR"] = str(cache_dir)

    root  = _find_dataset_root(args.data_root)
    names = _load_names(root / "data.yaml")
    K     = len(names)

    print(f"\nDataset root : {root}")
    print(f"Classes ({K}) : {names}")

    train_items = _gather_split(root, "train")
    valid_items = _gather_split(root, "valid")
    test_items  = _gather_split(root, "test")
    trainval    = train_items + valid_items

    if not trainval:
        raise SystemExit("No labelled train/valid images found.")

    print(f"Train        : {len(train_items)} images")
    print(f"Valid        : {len(valid_items)} images  (clean, zero leakage)")
    print(f"Test         : {len(test_items)} images  (clean, zero leakage)")
    print(f"Trainval pool: {len(trainval)} images  (pooled for CV fold splits)")

    if args.quick:
        random.Random(0).shuffle(trainval)
        trainval   = trainval[:300]
        test_items = test_items[:100]
        args.epochs = 3
        print("  [quick] smoke-test: 3 epochs, 300 trainval, 100 test")

    work = Path(args.work_dir).resolve()
    (work / "ensemble").mkdir(parents=True, exist_ok=True)

    Y          = _multi_hot(trainval, K)
    fold_of    = _repair_presence(
                     _stratified_kfold(Y, args.folds, args.cv_seed), Y, args.folds)
    fold_yamls = _write_fold_files(work, trainval, fold_of, args.folds, names)

    print(f"Fold val sizes: { {f: int((fold_of==f).sum()) for f in range(args.folds)} }")
    for f in range(args.folds):
        val_idx = np.where(fold_of == f)[0]
        cls_cnt = Counter(
            int(line.split()[0])
            for idx in val_idx
            for line in trainval[idx]["label"].read_text().splitlines()
            if line.strip()
        )
        print(f"  fold {f} val: { {names[c]: cnt for c, cnt in sorted(cls_cnt.items())} }")

    summary: dict[str, Any] = {
        "session":  SESSION_NAME,
        "model":    args.model,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset":  str(root),
        "dataset_sizes": {
            "train": len(train_items),
            "valid": len(valid_items),
            "test":  len(test_items),
        },
        "config": {k: getattr(args, k) for k in (
            "folds", "epochs", "imgsz", "batch", "device", "cv_seed", "seeds",
            "conf", "wbf_iou", "min_support", "copy_paste", "mixup",
            "label_smoothing", "mosaic", "close_mosaic")},
        "classes": names,
        "phaseA_cv": [], "phaseB_ensemble": {}, "phaseC_seeds": [],
        "phaseD_final": {},
    }
    summary_path = work / f"{SESSION_NAME}_summary.json"

    def _save():
        summary_path.write_text(json.dumps(summary, indent=2, default=str))

    # ==========================================================================
    # PHASE A - 5-fold cross-validation
    # ==========================================================================
    print("\n" + "="*64 + "\nPHASE A - 5-fold cross-validation\n" + "="*64)
    fold_models: list[str] = []
    args.seed_for_run = args.cv_seed

    if not args.skip_train:
        for f, fy in enumerate(fold_yamls):
            print(f"\n--- Fold {f}/{args.folds-1}  seed={args.cv_seed} ---")
            args.seed_for_run = args.cv_seed
            res = _train_fold(args.model, fy, f"fold_{f}_seed{args.cv_seed}", args)
            res["fold"] = f
            summary["phaseA_cv"].append(res)
            fold_models.append(res["best_pt"])
            print(f"  Fold {f}: val mAP50 = {res['val_map50']:.4f}  "
                  f"mAP50-95 = {res['val_map50_95']:.4f}")
            _save()
        if summary["phaseA_cv"]:
            arr = np.array([r["val_map50"] for r in summary["phaseA_cv"]], dtype=float)
            summary["phaseA_cv_mean_map50"] = float(np.nanmean(arr))
            summary["phaseA_cv_std_map50"]  = float(np.nanstd(arr))
            print(f"\nCV mask mAP50 = {np.nanmean(arr):.4f} +/- {np.nanstd(arr):.4f}")
            _save()
    else:
        for f in range(args.folds):
            bp = work / "runs" / f"fold_{f}_seed{args.cv_seed}" / "weights" / "best.pt"
            if bp.is_file():
                fold_models.append(str(bp))
        print(f"--skip-train: loaded {len(fold_models)} saved models.")

    # Real measured rate from THIS run, not a guess from a previous session -
    # used below to size/skip Phase B and Phase D against actual remaining time.
    phaseA_elapsed_min = _elapsed_min()
    per_epoch_min = (
        phaseA_elapsed_min / (args.folds * args.epochs)
        if (not args.skip_train and args.folds > 0 and args.epochs > 0)
        else None
    )
    print(f"\n  [budget] elapsed so far: {phaseA_elapsed_min:.1f} min "
          f"of {args.session_budget_min:.0f} min budget"
          + (f" (~{per_epoch_min:.2f} min/epoch measured)" if per_epoch_min else ""))

    # ==========================================================================
    # PHASE B - WBF ensemble on held-out test
    # ==========================================================================
    remaining_min = args.session_budget_min - _elapsed_min()
    if test_items and fold_models and remaining_min < args.phaseB_reserve_min:
        print(f"\n  [budget] skipping Phase B: only {remaining_min:.1f} min left "
              f"(need >= {args.phaseB_reserve_min:.0f} min reserve). Phase A's CV "
              f"result above is unaffected by this.")
        summary["phaseB_ensemble"] = {
            "skipped": True,
            "reason": f"only {remaining_min:.1f} min left in session budget",
        }
        _save()
    elif test_items and fold_models:
        print("\n" + "="*64 + "\nPHASE B - WBF fold ensemble on held-out test\n" + "="*64)
        # Extra safety net right before Phase B starts: Phase A just finished
        # 5 rounds of training, sweep GPU memory clean before the first
        # model.predict() call rather than trusting per-fold cleanup alone.
        _free_gpu_memory()

        gt_json    = work / "ensemble" / "coco_gt_test.json"
        stem_to_id = _build_coco_gt(test_items, names, gt_json)

        preds_by_model: list[dict[str, list]] = []
        single_scores:  list[float]           = []

        for f, bp in enumerate(fold_models):
            print(f"  predicting test with fold {f} ...")
            preds = _predict(bp, test_items, args)
            preds_by_model.append(preds)
            solo = {it["stem"]: _fuse_image([preds.get(it["stem"], [])], 1, args)
                    for it in test_items}
            dt   = _dets_to_coco(solo, stem_to_id,
                                 work / "ensemble" / f"coco_dt_fold{f}.json")
            m    = _coco_eval(gt_json, dt, names)
            single_scores.append(m["mAP50"])
            print(f"    fold {f} solo test mAP50 = {m['mAP50']:.4f}")

        print(f"  fusing {len(fold_models)} fold models ...")
        fused: dict[str, list] = {}
        for it in test_items:
            fused[it["stem"]] = _fuse_image(
                [pm.get(it["stem"], []) for pm in preds_by_model],
                len(fold_models), args)
        dt_ens = _dets_to_coco(fused, stem_to_id,
                               work / "ensemble" / "coco_dt_ensemble.json")
        ens    = _coco_eval(gt_json, dt_ens, names)
        mean_s = float(np.mean(single_scores)) if single_scores else None

        summary["phaseB_ensemble"] = {
            "solo_fold_test_map50":     single_scores,
            "solo_mean_test_map50":     mean_s,
            "ensemble_test_map50":      ens["mAP50"],
            "ensemble_test_map50_95":   ens["mAP50_95"],
            "ensemble_per_class_map50": ens["per_class_mAP50"],
            "ensemble_gain": (ens["mAP50"] - mean_s) if mean_s is not None else None,
        }
        print(f"\n  Solo-mean test mAP50 = {mean_s:.4f}")
        print(f"  ENSEMBLE test mAP50  = {ens['mAP50']:.4f}  "
              f"(gain {ens['mAP50']-(mean_s or 0):+.4f})")
        print(f"  mAP50-95             = {ens['mAP50_95']:.4f}")
        print("  Per-class mAP50:")
        for cls_name, v in ens["per_class_mAP50"].items():
            print(f"    {cls_name}: {v:.4f}")
        _save()

    # ==========================================================================
    # PHASE C - multi-seed stability on fold 0
    # ==========================================================================
    extra_seeds = [s for s in args.seeds if s != args.cv_seed]
    if extra_seeds and not args.skip_train and test_items:
        print("\n" + "="*64 + "\nPHASE C - multi-seed stability (fold 0)\n" + "="*64)
        gt_json = work / "ensemble" / "coco_gt_test.json"
        if not gt_json.is_file():
            stem_to_id = _build_coco_gt(test_items, names, gt_json)
        else:
            stem_to_id = {it["stem"]: i+1 for i, it in enumerate(test_items)}
        seed_rows: list[dict] = []
        for s in [args.cv_seed] + extra_seeds:
            args.seed_for_run = s
            run_name = f"fold_0_seed{s}"
            print(f"\n  seed {s} ...")
            res   = _train_fold(args.model, fold_yamls[0], run_name, args)
            preds = _predict(res["best_pt"], test_items, args)
            solo  = {it["stem"]: _fuse_image([preds.get(it["stem"], [])], 1, args)
                     for it in test_items}
            dt    = _dets_to_coco(solo, stem_to_id,
                                  work / "ensemble" / f"coco_dt_{run_name}.json")
            m     = _coco_eval(gt_json, dt, names)
            seed_rows.append({"seed": s, "val_map50": res["val_map50"],
                              "test_map50": m["mAP50"]})
            print(f"    seed {s}: val {res['val_map50']:.4f} | test {m['mAP50']:.4f}")
            summary["phaseC_seeds"] = seed_rows
            _save()
        tv = np.array([r["test_map50"] for r in seed_rows], dtype=float)
        summary["phaseC_seed_test_mean"] = float(np.nanmean(tv))
        summary["phaseC_seed_test_std"]  = float(np.nanstd(tv))
        print(f"\n  Seed test mAP50 = {np.nanmean(tv):.4f} +/- {np.nanstd(tv):.4f}")
        _save()

    # ==========================================================================
    # PHASE D - final model: train on ALL trainval, evaluate on held-out test
    # ==========================================================================
    if not args.skip_train and test_items:
        remaining_min = args.session_budget_min - _elapsed_min()
        trainable_min = remaining_min - args.phaseD_reserve_min

        if per_epoch_min is None:
            # Unknown rate (e.g. Phase A was skipped/loaded from disk) - can't
            # size Phase D against a measured speed, so use --phaseD-epochs
            # as given and just warn that timing isn't predictable here.
            epochs_to_use = args.phaseD_epochs
            print(f"\n  [budget] per-epoch rate unknown (Phase A wasn't trained this "
                  f"run) - running Phase D for the requested {epochs_to_use} epochs "
                  f"without a time guarantee.")
        else:
            phaseD_size_ratio = args.folds / max(1, args.folds - 1)  # trainval vs one fold's train subset
            phaseD_per_epoch_min = per_epoch_min * phaseD_size_ratio
            max_affordable_epochs = int(trainable_min // phaseD_per_epoch_min) if phaseD_per_epoch_min > 0 else 0
            epochs_to_use = max(0, min(args.phaseD_epochs, max_affordable_epochs))
            print(f"\n  [budget] {remaining_min:.1f} min left, ~{phaseD_per_epoch_min:.2f} "
                  f"min/epoch estimated for Phase D's larger dataset -> "
                  f"affordable epochs = {max_affordable_epochs}, using {epochs_to_use} "
                  f"(requested {args.phaseD_epochs}).")

        if epochs_to_use < 1:
            print(f"  [budget] skipping Phase D entirely: not enough time left for even "
                  f"1 fine-tune epoch. Your best Phase A fold is the model to use instead.")
            best_fold = max(summary["phaseA_cv"], key=lambda r: r["val_map50"]) if summary["phaseA_cv"] else None
            summary["phaseD_final"] = {
                "skipped": True,
                "reason": f"only {remaining_min:.1f} min left in session budget",
                "recommended_model": best_fold["best_pt"] if best_fold else None,
                "recommended_model_val_map50": best_fold["val_map50"] if best_fold else None,
            }
            _save()
        else:
            print("\n" + "="*64 + "\nPHASE D - final model on all trainval\n" + "="*64)
            print(f"  Training on all {len(trainval)} trainval images (no fold split) ...")

            final_dir = work / "folds" / "final"
            final_dir.mkdir(parents=True, exist_ok=True)
            all_txt = final_dir / "trainval.txt"
            all_txt.write_text(
                "\n".join(str(it["img"].resolve()) for it in trainval) + "\n")
            test_txt = final_dir / "test.txt"
            test_txt.write_text(
                "\n".join(str(it["img"].resolve()) for it in test_items) + "\n")
            final_dy = {
                "train": str(all_txt.resolve()),
                "val":   str(test_txt.resolve()),
                "nc":    len(names),
                "names": {i: n for i, n in enumerate(names)},
            }
            (final_dir / "data.yaml").write_text(yaml.safe_dump(final_dy, sort_keys=False))

            if summary["phaseA_cv"]:
                best_fold = max(summary["phaseA_cv"], key=lambda r: r["val_map50"])
                init_weights = best_fold["best_pt"]
                print(f"  Fine-tuning from best fold {best_fold['fold']} "
                      f"(val_map50={best_fold['val_map50']:.4f}): {init_weights}")
            else:
                init_weights = args.model
                print(f"  [warn] no Phase A results available, starting Phase D from "
                      f"base weights {init_weights} instead of fine-tuning.")

            args.seed_for_run = args.cv_seed
            final_res = _train_fold(
                init_weights,
                final_dir / "data.yaml",
                "final_all_trainval",
                args,
                epochs_override=epochs_to_use,
            )
            print(f"  Final model trained. Evaluating on held-out test ...")

            gt_json = work / "ensemble" / "coco_gt_test.json"
            if not gt_json.is_file():
                stem_to_id = _build_coco_gt(test_items, names, gt_json)
            else:
                stem_to_id = {it["stem"]: i+1 for i, it in enumerate(test_items)}

            final_preds = _predict(final_res["best_pt"], test_items, args)
            final_solo  = {it["stem"]: _fuse_image([final_preds.get(it["stem"], [])], 1, args)
                           for it in test_items}
            dt_final = _dets_to_coco(
                final_solo, stem_to_id,
                work / "ensemble" / "coco_dt_final.json")
            final_eval = _coco_eval(gt_json, dt_final, names)

            summary["phaseD_final"] = {
                "best_pt":               final_res["best_pt"],
                "trainval_size":         len(trainval),
                "phaseD_epochs_used":    epochs_to_use,
                "test_map50":            final_eval["mAP50"],
                "test_map50_95":         final_eval["mAP50_95"],
                "per_class_map50":       final_eval["per_class_mAP50"],
            }
            print(f"\n  FINAL MODEL test mAP50   = {final_eval['mAP50']:.4f}")
            print(f"  FINAL MODEL mAP50-95     = {final_eval['mAP50_95']:.4f}")
            print("  Per-class mAP50:")
            for cls_name, v in final_eval["per_class_mAP50"].items():
                print(f"    {cls_name}: {v:.4f}")
            _save()


    _save()
    print("\n" + "="*64 + "\nSESSION 18 COMPLETE\n" + "="*64)
    if "phaseA_cv_mean_map50" in summary:
        print(f"CV (val) mAP50       : {summary['phaseA_cv_mean_map50']:.4f} "
              f"+/- {summary['phaseA_cv_std_map50']:.4f}")
    if summary["phaseB_ensemble"]:
        b = summary["phaseB_ensemble"]
        if b.get("solo_mean_test_map50") is not None:
            print(f"Test solo-mean mAP50 : {b['solo_mean_test_map50']:.4f}")
        print(f"Test ENSEMBLE mAP50  : {b['ensemble_test_map50']:.4f}  "
              f"(gain {b.get('ensemble_gain', 0):+.4f})")
        print(f"Test mAP50-95        : {b['ensemble_test_map50_95']:.4f}")
    if summary.get("phaseC_seeds"):
        print(f"Seed stability       : {summary['phaseC_seed_test_mean']:.4f} "
              f"+/- {summary['phaseC_seed_test_std']:.4f}")
    if summary.get("phaseD_final"):
        d = summary["phaseD_final"]
        print(f"FINAL MODEL mAP50    : {d['test_map50']:.4f}  <- production model (all trainval)")
        print(f"FINAL MODEL mAP50-95 : {d['test_map50_95']:.4f}")
    print(f"\nSummary JSON : {summary_path}")
    print(f"Models dir   : {work / 'runs'}")
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    _rc = main()
    try:
        sys.exit(_rc)
    except SystemExit as _e:
        if _e.code:
            raise

  [mem] PYTORCH_ALLOC_CONF = expandable_segments:True
  [install] ultralytics ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.5 MB/s eta 0:00:00
  [install] iterative-stratification ...
  [device] 2 GPUs -> '0,1' (DDP)
  [batch] auto -> 16 (2 GPU(s) x 8 - OOM-safe for T4)
  [auto-detect] found dataset layout at: /kaggle/input/datasets/abrarfaiyazsifat/image-annotation-rebuilt

Dataset root : /kaggle/input/datasets/abrarfaiyazsifat/image-annotation-rebuilt
Classes (4) : ['Nail_psoriasis', 'SJS-TEN', 'acne', 'vitiligo']
Train        : 6057 images
Valid        : 1085 images  (clean, zero leakage)
Test         : 1030 images  (clean, zero leakage)
Trainval pool: 7142 images  (pooled for CV fold splits)
Fold val sizes: {0: 1428, 1: 1429, 2: 1428, 3: 1428, 4: 1429}
  fold 0 val: {'Nail_psoriasis': 1477, 'SJS-TEN': 568, 'acn

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/48         6G      1.317      2.651        1.8      1.661          0         18        640: 100% ━━━━━━━━━━━━ 304/304 2.4it/s 2:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 3.1it/s 14.5s
                   all       1428       2844      0.419      0.361      0.296      0.157      0.397       0.34      0.266      0.136

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/48         6G      1.262      2.553      1.705      1.616          0         18        640: 100% ━━━━━━━━━━━━ 304/304 2.5it/s 2:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 3.1it/s 14.3s
                   all       1428       2

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/48      5.98G      1.322      2.618      1.762      1.641          0         27        640: 100% ━━━━━━━━━━━━ 304/304 2.4it/s 2:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 3.0it/s 14.8s
                   all       1429       2877      0.486      0.426      0.361       0.18      0.453      0.389      0.317      0.145

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/48      6.04G      1.275      2.569      1.669      1.612          0         58        640: 100% ━━━━━━━━━━━━ 304/304 2.4it/s 2:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 3.2it/s 14.3s
                   all       1429       2

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/48         6G      1.302      2.609      1.753      1.646          0         21        640: 100% ━━━━━━━━━━━━ 304/304 2.4it/s 2:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 3.2it/s 14.1s
                   all       1429       2970      0.445       0.38      0.337      0.164      0.404      0.349      0.288       0.13

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/48      6.02G      1.269      2.534      1.662      1.624          0         51        640: 100% ━━━━━━━━━━━━ 304/304 2.4it/s 2:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 3.0it/s 14.8s
                   all       1429       2

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/12      6.02G     0.8927      1.956     0.9698      1.337          0         29        640: 100% ━━━━━━━━━━━━ 380/380 2.4it/s 2:37
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 3.2it/s 10.4s
                   all       1030       1701      0.748      0.666      0.687      0.439      0.717      0.641      0.642      0.365
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/12      6.04G      1.174      2.327      1.183      1.672          0         10        640: 100% ━━━━━━━